In [1]:
import numpy as np
import scipy as sp
from LQRMSD import LQRMSD

In [ ]:
Ts = 0.5  # or 0.05
Tsim = 6  # must be multiple of Ts

nx = 2
nu = 1

# penalties
Qe = np.eye(nx) * 2  # state cost
Re = np.eye(nu) * 10  # input cost
# terminal state cost, needs to be 0 for the extended state since
# we dont want to penalize the state being away from the origin
Pe = np.eye(nx) * 0

x0 = np.array([[1], [0]])
x0_mean = x0
x0_cov = np.eye(nx) * 0.1
ref = np.array([[2], [0]])  # reference state [pos, vel]
u0 = np.array([[0]])  # initial control effort !=0 to prevent repmat warnings
x0_ext = np.vstack([x0, u0, ref])  # extended state
x0_ext_mean = np.vstack([x0_mean, u0, ref])
x0_ext_cov = sp.linalg.block_diag(x0_cov, np.zeros(
    (u0.shape[0], u0.shape[0])), np.zeros((ref.shape[0], ref.shape[0])))

rv_samples = 10000
x0_rv = np.random.multivariate_normal(x0_mean.flatten(), x0_cov, rv_samples).T
x0_rv_mean = np.mean(x0_rv, axis=1).reshape(-1, 1)
x0_rv_cov = np.cov(x0_rv)
x0_rv_ext = np.vstack([x0_rv, np.tile(u0, (1, rv_samples)), np.tile(ref, (1, rv_samples))])
x0_rv_ext_mean = np.vstack([x0_rv_mean, u0, ref])
x0_rv_ext_cov = sp.linalg.block_diag(x0_rv_cov, np.zeros(
    (u0.shape[0], u0.shape[0])), np.zeros((ref.shape[0], ref.shape[0])))

In [ ]:
lqr_prob = LQRMSD(Ts, Tsim, Qe, Re, Pe)

Uopt_det = lqr_prob.Kopt @ x0_ext_mean
obj_det = lqr_prob.LQRObj(x0_ext, Uopt_det)
print(f"Deterministic LQR objective (LQRMSD): {obj_det}")

Uopt_stoch = lqr_prob.Kopt @ x0_rv_ext_mean
obj_stoch = lqr_prob.LQRObj(x0_rv_ext, Uopt_stoch)
obj_exp = lqr_prob.LQRExp(x0_rv_ext_mean, x0_rv_ext_cov, Uopt_stoch)
obj_mean_st = np.mean(obj_stoch)
print(f"LQR expectation (LQRMSD):\n  Analytical: {obj_exp}\n  Statistical: {obj_mean_st}")
obj_var = lqr_prob.LQRVar(x0_rv_ext_mean, x0_rv_ext_cov, Uopt_stoch)
obj_var_st = np.var(obj_stoch)
print(f"LQR variance (LQRMSD):\n  Analytical: {obj_var}\n  Statistical: {obj_var_st}")

Deterministic LQR objective (LQRMSD): [[40.02958062]]
LQR expectation (LQRMSD):
  Analytical: [[44.61165269]]
  Statistical: 44.611206454458056
LQR variance (LQRMSD):
  Analytical: [[207.63839463]]
  Statistical: 207.5290644823531
